<a href="https://colab.research.google.com/github/diegovianagomes/mestrado/blob/main/WD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install neurokit2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from scipy import signal
from scipy.signal import butter, filtfilt, savgol_filter
from scipy.stats import uniform, randint

import neurokit2 as nk
import os

from pathlib import Path
import pickle

from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedKFold, cross_validate, GroupKFold, RandomizedSearchCV
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


from xgboost import XGBClassifier
import xgboost as xgb

In [ ]:
np.random.seed(42)

In [ ]:
base_path = "/content/drive/MyDrive/03_MESTRADO/DIEGO_VIANA_PROJETO_DE_PESQUISA/Pesquisa Datasets/wearable_stress_exercise/data/Wearable_Dataset"
metadata_path = "/content/drive/MyDrive/03_MESTRADO/DIEGO_VIANA_PROJETO_DE_PESQUISA/Pesquisa Datasets/wearable_stress_exercise/data"


# Função para carregar dados

In [ ]:
def load_participant_data(participant_id, category='STRESS'):
    participant_path = f"{base_path}/{category}/{participant_id}"
    data = {}

    for file in ['ACC.csv', 'BVP.csv', 'EDA.csv', 'HR.csv', 'tags.csv']:
        file_path = os.path.join(participant_path, file)
        if not os.path.exists(file_path):
            print(f"Nao encontrado: {file}")
            continue

        try:
            with open(file_path, 'r') as f:
                lines = f.readlines()

            if file == 'ACC.csv':
                first_line = lines[0].strip().split(',')[0]
                sample_rate = float(lines[1].strip().split(',')[0])
                df = pd.read_csv(file_path, skiprows=2, header=None).iloc[:, :3]
                df.columns = ['x', 'y', 'z']

            elif file == 'tags.csv':
                first_line = lines[0].strip()
                start_time = datetime.strptime(first_line, '%Y-%m-%d %H:%M:%S').timestamp()
                timestamps = []
                for line in lines[1:]:
                    if line.strip():
                        tag_time = datetime.strptime(line.strip(), '%Y-%m-%d %H:%M:%S').timestamp()
                        timestamps.append(tag_time)
                df = pd.DataFrame({'timestamp': timestamps})
                data[file.replace('.csv', '')] = {
                    'data': df,
                    'sample_rate': None,
                    'start_time': start_time
                }
                print(f"tags.csv: {len(df)} eventos")
                continue

            else:
                first_line = lines[0].strip()
                sample_rate = float(lines[1].strip())
                df = pd.read_csv(file_path, skiprows=2, header=None)
                df.columns = ['value']

            if file != 'tags.csv':
                start_time = datetime.strptime(first_line, '%Y-%m-%d %H:%M:%S').timestamp()
                timestamps = start_time + np.arange(len(df)) / sample_rate
                df['timestamp'] = timestamps

            data[file.replace('.csv', '')] = {
                'data': df,
                'sample_rate': sample_rate,
                'start_time': start_time
            }

            print(f"{file}: {len(df)} amostras, {sample_rate}Hz")

        except Exception as e:
            print(f"Erro {file}: {e}")

    return data

# Função para extrair segmentos

In [ ]:
def extrair_segmento(sinal_data, inicio, fim):
    df = sinal_data['data']
    mascara = (df['timestamp'] >= inicio) & (df['timestamp'] <= fim)
    return df[mascara]

# processamento de sinais

In [ ]:
def processar_eda(eda_sinal, taxa_amostragem):
    try:
        if hasattr(eda_sinal, 'values'):
            eda_sinal = eda_sinal.values

        nyquist = taxa_amostragem / 2
        corte = 1.99 / nyquist
        b, a = butter(5, corte, btype='low')
        eda_filtrado = filtfilt(b, a, eda_sinal)

        janela = int(5 * taxa_amostragem)
        if janela % 2 == 0:
            janela += 1
        if janela > len(eda_filtrado):
            janela = len(eda_filtrado) if len(eda_filtrado) % 2 == 1 else len(eda_filtrado) - 1

        tonic = savgol_filter(eda_filtrado, janela, 2)
        phasic = eda_filtrado - tonic

        return eda_filtrado, tonic, phasic
    except Exception as e:
        print(f"Erro processar_eda: {e}")
        return eda_sinal, eda_sinal, np.zeros_like(eda_sinal)


#TODO:

In [ ]:
def processar_bvp(bvp_sinal, taxa_amostragem):
    try:
        if hasattr(bvp_sinal, 'values'):
            bvp_sinal = bvp_sinal.values

        bvp_normalizado = (bvp_sinal - np.mean(bvp_sinal)) / np.std(bvp_sinal)

        bvp_filtrado = nk.signal_filter(bvp_normalizado, sampling_rate=taxa_amostragem,
                                       lowcut=0.5, highcut=4.0, method="butterworth")


        sinais, info = nk.ppg_process(bvp_filtrado, sampling_rate=taxa_amostragem)

        picos = info['PPG_Peaks']

        if picos.sum() > 10:
            indices_picos = np.where(picos)[0]
            intervalos = np.diff(indices_picos) / taxa_amostragem
            intervalos_filtrados = intervalos[(intervalos > 0.3) & (intervalos < 2.0)]

            if len(intervalos_filtrados) > 5:
                return picos, intervalos_filtrados
            else:
                print(f"  Intervalos após filtro: {len(intervalos_filtrados)} (insuficientes)")
                return picos, np.array([])
        else:
            print(f"  Poucos picos detectados: {picos.sum()}")
            return picos, np.array([])

    except Exception as e:
        print(f"Erro processar_bvp: {e}")
        return np.array([]), np.array([])


for nome_bloco, dados_bloco in features_por_bloco.items():
    segmentos = dados_bloco['segmentos']
    if 'BVP' in segmentos:
        bvp_sinal = segmentos['BVP']['value'].values
        taxa_bvp = example_data['BVP']['sample_rate']

        print(f"\n{nome_bloco}:")
        picos, intervalos = processar_bvp(bvp_sinal, taxa_bvp)

        print(f"  Picos detectados: {picos.sum() if picos is not None else 0}")
        print(f"  Intervalos válidos: {len(intervalos)}")

        if len(intervalos) > 5:
            print(f"  Intervalos (primeiros 5): {intervalos[:5]}")
            print(f"  Frequência cardíaca média: {60/np.mean(intervalos):.1f} bpm")

# extração de features

In [ ]:
def extrair_features_eda(eda_sinal, taxa_amostragem):
    try:
        eda_filtrado, tonic, phasic = processar_eda(eda_sinal, taxa_amostragem)

        features = {}
        features['mean_raw_eda'] = np.mean(eda_sinal)
        features['std_raw_eda'] = np.std(eda_sinal)
        features['mean_tonic_eda'] = np.mean(tonic)
        features['std_tonic_eda'] = np.std(tonic)
        features['mean_phasic_eda'] = np.mean(phasic)
        features['std_phasic_eda'] = np.std(phasic)

        derivada_tonic = np.diff(tonic)
        if len(derivada_tonic) > 0:
            features['tonic_ratio_down'] = np.percentile(derivada_tonic, 5)
            features['tonic_ratio_up'] = np.percentile(derivada_tonic, 95)
        else:
            features['tonic_ratio_down'] = 0
            features['tonic_ratio_up'] = 0

        try:
            scr_signals, scr_info = nk.eda_peaks(tonic, sampling_rate=taxa_amostragem)
            peaks_count = scr_info['SCR_Peaks'].sum()
            features['peaks_density'] = peaks_count / (len(tonic) / taxa_amostragem)

            if peaks_count > 0:
                features['scr_mean_amp'] = scr_info['SCR_Amplitude'][scr_info['SCR_Amplitude'] > 0].mean()
                features['scr_mean_height'] = scr_info['SCR_Height'][scr_info['SCR_Height'] > 0].mean()
                features['scr_mean_risetime'] = scr_info['SCR_RiseTime'][scr_info['SCR_RiseTime'] > 0].mean()
                features['scr_mean_recoverytime'] = scr_info['SCR_RecoveryTime'][scr_info['SCR_RecoveryTime'] > 0].mean()
            else:
                features['scr_mean_amp'] = 0
                features['scr_mean_height'] = 0
                features['scr_mean_risetime'] = 0
                features['scr_mean_recoverytime'] = 0
        except Exception as e:
            print(f"Erro EDA peaks: {e}")
            features['peaks_density'] = 0
            features['scr_mean_amp'] = 0
            features['scr_mean_height'] = 0
            features['scr_mean_risetime'] = 0
            features['scr_mean_recoverytime'] = 0

        return features
    except Exception as e:
        print(f"Erro extrair_features_eda: {e}")
        return {}


In [ ]:
def extrair_features_hr_com_variabilidade(hr_sinal):
    try:
        features = {
            'hr_mean': np.mean(hr_sinal),
            'hr_std': np.std(hr_sinal),
            'hr_variability': np.std(hr_sinal)
        }

        derivada_hr = np.diff(hr_sinal)
        if len(derivada_hr) > 0:
            features['hr_ratio_down'] = np.percentile(derivada_hr, 5)
            features['hr_ratio_up'] = np.percentile(derivada_hr, 95)
        else:
            features['hr_ratio_down'] = 0
            features['hr_ratio_up'] = 0

        return features
    except Exception as e:
        print(f"Erro extrair_features_hr: {e}")
        return {
            'hr_mean': 0,
            'hr_std': 0,
            'hr_variability': 0,
            'hr_ratio_down': 0,
            'hr_ratio_up': 0
        }

In [ ]:
def extrair_features_bvp(bvp_sinal, taxa_amostragem):
    try:
        features = {
            'bvp_mean': np.mean(bvp_sinal),
            'bvp_std': np.std(bvp_sinal)
        }
        return features
    except Exception as e:
        print(f"Erro extrair_features_bvp: {e}")
        return {
            'bvp_mean': 0,
            'bvp_std': 0
        }

In [ ]:
def extrair_features_acc(acc_data, taxa_amostragem):
    try:
        x = acc_data['x'].values
        y = acc_data['y'].values
        z = acc_data['z'].values

        magnitude = np.sqrt(x**2 + y**2 + z**2)

        features = {
            'x_mean': np.mean(x),
            'x_std': np.std(x),
            'y_mean': np.mean(y),
            'y_std': np.std(y),
            'z_mean': np.mean(z),
            'z_std': np.std(z),
            'acc_mean': np.mean(magnitude),
            'acc_std': np.std(magnitude)
        }

        derivada_mag = np.diff(magnitude)
        if len(derivada_mag) > 0:
            features['acc_ratio_down'] = np.percentile(derivada_mag, 5)
            features['acc_ratio_up'] = np.percentile(derivada_mag, 95)
        else:
            features['acc_ratio_down'] = 0
            features['acc_ratio_up'] = 0

        return features
    except Exception as e:
        print(f"Erro extrair_features_acc: {e}")
        return {}

In [ ]:
def extrair_features_hr(hr_sinal):
    try:
        features = {
            'hr_mean': np.mean(hr_sinal),
            'hr_std': np.std(hr_sinal),
            'hr_variability': np.std(hr_sinal)
        }

        derivada_hr = np.diff(hr_sinal)
        if len(derivada_hr) > 0:
            features['hr_ratio_down'] = np.percentile(derivada_hr, 5)
            features['hr_ratio_up'] = np.percentile(derivada_hr, 95)
        else:
            features['hr_ratio_down'] = 0
            features['hr_ratio_up'] = 0

        return features
    except Exception as e:
        print(f"Erro extrair_features_hr: {e}")
        return {
            'hr_mean': 0,
            'hr_std': 0,
            'hr_variability': 0,
            'hr_ratio_down': 0,
            'hr_ratio_up': 0
        }


In [ ]:
def extrair_features_hrv_do_hr(hr_sinal):

    try:
        hr_diff = np.diff(hr_sinal)

        features = {
            'hr_mean': np.mean(hr_sinal),
            'hr_std': np.std(hr_sinal),
            'hr_variability': np.std(hr_diff) if len(hr_diff) > 0 else 0,
            'hr_range': np.max(hr_sinal) - np.min(hr_sinal)
        }
        return features

    except Exception as e:
        print(f"Erro extrair_features_hrv_do_hr: {e}")
        return {
            'hr_mean': 0,
            'hr_std': 0,
            'hr_variability': 0,
            'hr_range': 0
        }

In [ ]:
participant_example = 'S01'
example_data = load_participant_data(participant_example, 'STRESS')

for signal_type in example_data.keys():
    print(f"{signal_type}: {len(example_data[signal_type]['data'])} amostras")

# Segmentação baseado nas tags

In [ ]:
tags_df = example_data['tags']['data']
tags_times = tags_df['timestamp'].values
start_time = example_data['ACC']['start_time']

- Define os blocos do protocolo v1

In [ ]:
blocos = {
    'Baseline': (tags_times[0], tags_times[1]),
    'Stroop': (tags_times[1], tags_times[2]),
    'First_Rest': (tags_times[2], tags_times[3]),
    'TMCT': (tags_times[3], tags_times[4]),
    'Second_Rest': (tags_times[4], tags_times[5]),
    'Real_Opinion': (tags_times[5], tags_times[6]),
    'Opposite_Opinion': (tags_times[6], tags_times[7]),
    'Subtract': (tags_times[7], tags_times[8])
}

for nome, (inicio, fim) in blocos.items():
    duracao = (fim - inicio) / 60
    print(f"  {nome}: {duracao:.2f} min")

- Dicionário para armazenar dados segmentados

In [ ]:
features_por_bloco = {}

for nome_bloco, (inicio, fim) in blocos.items():
    segmentos = {}
    for sinal in ['ACC', 'BVP', 'EDA', 'HR']:
        if sinal in example_data:
            segmento = extrair_segmento(example_data[sinal], inicio, fim)
            segmentos[sinal] = segmento
    features_por_bloco[nome_bloco] = {
        'segmentos': segmentos,
        'inicio': inicio,
        'fim': fim
    }
    print(f"{nome_bloco}: {len(segmentos)} sinais extraídos")

# Extrair features para cada bloco

In [ ]:
for nome_bloco, dados_bloco in features_por_bloco.items():
    if 'features' not in dados_bloco:
        features_por_bloco[nome_bloco]['features'] = {}
    print(f"{nome_bloco}")

In [ ]:
for nome_bloco, dados_bloco in features_por_bloco.items():
    print(f"Processando {nome_bloco}...")
    segmentos = dados_bloco['segmentos']
    features_bloco = {}

    if 'EDA' in segmentos:
        taxa_eda = example_data['EDA']['sample_rate']
        features_eda = extrair_features_eda(segmentos['EDA']['value'].values, taxa_eda)
        features_bloco.update(features_eda)

    if 'BVP' in segmentos:
        taxa_bvp = example_data['BVP']['sample_rate']
        features_bvp = extrair_features_bvp(segmentos['BVP']['value'].values, taxa_bvp)
        features_bloco.update(features_bvp)


    if 'ACC' in segmentos:
        taxa_acc = example_data['ACC']['sample_rate']
        features_acc = extrair_features_acc(segmentos['ACC'], taxa_acc)
        features_bloco.update(features_acc)

    if 'HR' in segmentos:
        features_hr = extrair_features_hrv_do_hr(segmentos['HR']['value'].values)
        features_bloco.update(features_hr)

    features_por_bloco[nome_bloco]['features'] = features_bloco
    print(f"  Features extraídas: {len(features_bloco)}")

# DataFrame com features e labels

In [ ]:
blocos_labels = {
    'Baseline': 0,
    'Stroop': 1,
    'First_Rest': 0,
    'TMCT': 1,
    'Second_Rest': 0,
    'Real_Opinion': 1,
    'Opposite_Opinion': 1,
    'Subtract': 1
}

for nome_bloco in features_por_bloco.keys():
    if nome_bloco in blocos_labels:
        features_por_bloco[nome_bloco]['label'] = blocos_labels[nome_bloco]
        print(f"{nome_bloco}: label {features_por_bloco[nome_bloco]['label']}")

# Datafrane
linhas = []
for nome_bloco, dados_bloco in features_por_bloco.items():
    if 'features' in dados_bloco and 'label' in dados_bloco:
        linha = dados_bloco['features'].copy()
        linha['bloco'] = nome_bloco
        linha['label'] = dados_bloco['label']
        linha['participante'] = 'S01'
        linhas.append(linha)

df_features = pd.DataFrame(linhas)
df_features = df_features.fillna(0)

display(df_features.shape)
print(f"Blocos: {df_features['bloco'].tolist()}")
print(f"Labels: {df_features['label'].tolist()}")
print(f"Colunas: {df_features.columns.tolist()}")

# dados para modelagem com S01

In [ ]:
X = df_features.drop(['bloco', 'participante', 'label'], axis=1)
y = df_features['label']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Features: {list(X.columns)}")
print(f"Valores de y: {y.tolist()}")
print(f"Distribuição de y: {y.value_counts().to_dict()}")

- Selecionar features específicas do artigo

In [ ]:
df_features.head(8)

- Validação cruzada

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = XGBClassifier(random_state=42, eval_metric='logloss')
scoring = ['accuracy', 'precision', 'recall', 'f1']

cv_results = cross_validate(model, X, y, cv=cv, scoring=scoring, return_train_score=False)

metrics = {}
for metric in scoring:
    scores = cv_results[f'test_{metric}']
    metrics[metric] = {
        'mean': np.mean(scores) * 100,
        'std': np.std(scores) * 100
    }

for metric, values in metrics.items():
    print(f"{metric.upper():10} : {values['mean']:5.1f}% ± {values['std']:4.1f}%")

In [ ]:
for i in range(5):
    print(f"Fold {i+1}: ", end="")
    for metric in scoring:
        score = cv_results[f'test_{metric}'][i] * 100
        print(f"{metric}={score:.1f}%", end=", ")
    print()


# Como todos do Protocolo v1

- Lista de todos os participantes do v1

In [ ]:
v1_participants = [f'S{i:02d}' for i in range(1, 19)]
v1_participants

In [ ]:
def extrair_features_bvp_hrv(bvp_sinal, taxa_amostragem):

    features = {}

    features['bvp_mean'] = np.mean(bvp_sinal)
    features['bvp_std'] = np.std(bvp_sinal)

    try:
        bvp_clean = nk.ppg_clean(bvp_sinal, sampling_rate=taxa_amostragem)

        peaks_info = nk.ppg_findpeaks(bvp_clean, sampling_rate=taxa_amostragem)
        rr_intervals_ms = peaks_info['PPG_RR_Intervals_Ms']

        if len(rr_intervals_ms) > 0:
            ibi_seconds = rr_intervals_ms / 1000.0
            features['max_ibi'] = np.max(ibi_seconds)
            features['mean_ibi'] = np.mean(ibi_seconds)
            features['rmssd'] = nk.hrv_time(rr_intervals_ms, sampling_rate=taxa_amostragem)['RMSSD'][0]

            hrv_freq = nk.hrv_frequency(rr_intervals_ms, sampling_rate=taxa_amostragem, show=False)
            features['ratio'] = hrv_freq['LF/HF'][0]
            features['LF_peak'] = hrv_freq['LF_Peak'][0]

        else:
            for feat in ['max_ibi', 'mean_ibi', 'rmssd', 'ratio', 'LF_peak']:
                features[feat] = 0

    except Exception as e:
        print(f"Erro ao extrair HRV do BVP: {e}")
        for feat in ['max_ibi', 'mean_ibi', 'rmssd', 'ratio', 'LF_peak']:
            features[feat] = 0

    return features

In [ ]:
v1_participants = [f'S{i:02d}' for i in range(1, 19)]
all_dfs = []

In [ ]:
for participant_id in tqdm(v1_participants, desc="Processando v1"):
    try:

        example_data = load_participant_data(participant_id, 'STRESS')

        tags_df = example_data['tags']['data']
        tags_times = tags_df['timestamp'].values
        start_time = example_data['ACC']['start_time']

        blocos = {
            'Baseline': (tags_times[0], tags_times[1]), 'Stroop': (tags_times[1], tags_times[2]),
            'First_Rest': (tags_times[2], tags_times[3]), 'TMCT': (tags_times[3], tags_times[4]),
            'Second_Rest': (tags_times[4], tags_times[5]), 'Real_Opinion': (tags_times[5], tags_times[6]),
            'Opposite_Opinion': (tags_times[6], tags_times[7]), 'Subtract': (tags_times[7], tags_times[8])
        }

        features_por_bloco = {}
        for nome_bloco, (inicio, fim) in blocos.items():
            segmentos = {}
            for sinal in ['ACC', 'BVP', 'EDA', 'HR']:
                if sinal in example_data:
                    segmento = extrair_segmento(example_data[sinal], inicio, fim)
                    segmentos[sinal] = segmento
            features_por_bloco[nome_bloco] = {'segmentos': segmentos, 'inicio': inicio, 'fim': fim}

        blocos_labels = {'Baseline': 0, 'Stroop': 1, 'First_Rest': 0, 'TMCT': 1, 'Second_Rest': 0, 'Real_Opinion': 1, 'Opposite_Opinion': 1, 'Subtract': 1}
        linhas_do_participante = []

        for nome_bloco, dados_bloco in features_por_bloco.items():
            segmentos = dados_bloco['segmentos']
            features_bloco = {}

            if 'EDA' in segmentos:
                taxa_eda = example_data['EDA']['sample_rate']
                features_eda = extrair_features_eda(segmentos['EDA']['value'].values, taxa_eda)
                features_bloco.update(features_eda)

            if 'BVP' in segmentos:
                taxa_bvp = example_data['BVP']['sample_rate']
                features_bvp_hrv = extrair_features_bvp_hrv(segmentos['BVP']['value'].values, taxa_bvp)
                features_bloco.update(features_bvp_hrv)

            if 'ACC' in segmentos:
                taxa_acc = example_data['ACC']['sample_rate']
                features_acc = extrair_features_acc(segmentos['ACC'], taxa_acc)
                features_bloco.update(features_acc)

            if 'HR' in segmentos:
                features_hr = extrair_features_hrv_do_hr(segmentos['HR']['value'].values)
                features_bloco.update(features_hr)

            linha = features_bloco.copy()
            linha['bloco'] = nome_bloco
            linha['label'] = blocos_labels[nome_bloco]
            linha['participante'] = participant_id
            linhas_do_participante.append(linha)

        df_participante = pd.DataFrame(linhas_do_participante)
        all_dfs.append(df_participante)
        print(f"Processado com sucesso: {participant_id}")

    except Exception as e:
        print(f"Erro ao processar o participante {participant_id}: {e}.")
        continue



In [ ]:
df_full_v1 = pd.concat(all_dfs, ignore_index=True)
df_full_v1 = df_full_v1.fillna(0)

print(f"\nDataset final criado!")
print(f"Shape: {df_full_v1.shape}")
print(f"Distribuição de labels:")
print(df_full_v1['label'].value_counts())

In [ ]:
output_path = "/content/drive/MyDrive/03_MESTRADO/DIEGO_VIANA_PROJETO_DE_PESQUISA/Pesquisa Datasets/wearable_stress_exercise/data/processed"
df_full_v1.to_csv(f'{output_path}/STRESS_v1_features.csv', index=False)
print(f"Dataset salvo em: {output_path}/STRESS_v1_features.csv")

In [ ]:
display(df_full_v1.shape)
df_full_v1.head(50)

In [ ]:
X = df_full_v1.drop(['max_ibi',	'mean_ibi',	'rmssd',	'ratio',	'LF_peak', 'bloco', 'participante', 'label'], axis=1)
y = df_full_v1['label']

display(X.shape[1])
print(f"Features: {list(X.columns)}")

groups = df_full_v1['participante']
cv = GroupKFold(n_splits=5)

model = XGBClassifier(random_state=42, eval_metric='logloss')
scoring = ['accuracy', 'precision', 'recall', 'f1']

cv_results = cross_validate(model, X, y, groups=groups, cv=cv, scoring=scoring, return_train_score=False)

n_folds = cv.get_n_splits()
for i in range(n_folds):
    print(f"Fold {i+1}: ", end="")
    for metric in scoring:
        score = cv_results[f'test_{metric}'][i] * 100
        print(f"{metric}={score:.1f}%", end=", ")
    print()


metrics = {}
for metric in scoring:
    scores = cv_results[f'test_{metric}']
    metrics[metric] = {'mean': np.mean(scores) * 100, 'std': np.std(scores) * 100}

for metric, values in metrics.items():
    print(f"{metric.upper():10} : {values['mean']:5.1f}% ± {values['std']:4.1f}%")

To reduce inter-subject variability, intra-participant normalization was applied to all blocks within the stress protocol.

In [ ]:
df_normalized = df_full_v1.copy()

feature_cols = [col for col in df_normalized.columns if col not in ['max_ibi',	'mean_ibi',	'rmssd',	'ratio',	'LF_peak', 'bloco', 'participante', 'label']]

for participant in df_normalized['participante'].unique():
    mask = df_normalized['participante'] == participant
    participant_data = df_normalized.loc[mask, feature_cols]

    mean = participant_data.mean()
    std = participant_data.std()
    std[std == 0] = 1

    df_normalized.loc[mask, feature_cols] = (participant_data - mean) / std

display(df_normalized.head(30))

In [ ]:
X = df_normalized.drop(['max_ibi',	'mean_ibi',	'rmssd',	'ratio',	'LF_peak', 'bloco', 'participante', 'label'], axis=1)
y = df_normalized['label']

display(X.shape[1])
print(f"Features: {list(X.columns)}")

In [ ]:
groups = df_normalized['participante']
cv = GroupKFold(n_splits=5)

model = XGBClassifier(random_state=42, eval_metric='logloss')
gkf = GroupKFold(n_splits=5)

y_true_all = []
y_pred_all = []

In [ ]:
for train_index, test_index in gkf.split(X, y, groups):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_true_all.extend(y_test)
    y_pred_all.extend(y_pred)


cm = confusion_matrix(y_true_all, y_pred_all)

class_labels = ['Descanso (0)', 'Estresse (1)']


disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)

fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(cmap=plt.cm.Blues, ax=ax, values_format='d')

ax.set_title('Matriz de Confusão', fontsize=16)
ax.set_xlabel('Previsto ', fontsize=12)
ax.set_ylabel('Verdadeiro', fontsize=12)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")

plt.grid(False)
plt.show()


tn, fp, fn, tp = cm.ravel()

print(f"\nVerdadeiros Negativos (TN): {tn} - Previu 'Descanso' corretamente.")
print(f"Falsos Positivos (FP): {fp} - Previu 'Estresse', mas era 'Descanso'.")
print(f"Falsos Negativos (FN): {fn} - Previu 'Descanso', mas era 'Estresse'.")
print(f"Verdadeiros Positivos (TP): {tp} - Previu 'Estresse' corretamente.")

accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0


print(f"\nAcurácia: {accuracy:.2%}")
print(f"Precisão: {precision:.2%}")
print(f"Recall: {recall:.2%}")

In [ ]:
cv_results = cross_validate(model, X, y, groups=groups, cv=cv, scoring=scoring, return_train_score=False)

n_folds = cv.get_n_splits()
for i in range(n_folds):
    print(f"Fold {i+1}: ", end="")
    for metric in scoring:
        score = cv_results[f'test_{metric}'][i] * 100
        print(f"{metric}={score:.1f}%", end=", ")
    print()

metrics = {}
for metric in scoring:
    scores = cv_results[f'test_{metric}']
    metrics[metric] = {'mean': np.mean(scores) * 100, 'std': np.std(scores) * 100}

In [ ]:
groups = df_normalized['participante']

splits_dict = {}

for i, (train_index, test_index) in enumerate(gkf.split(X, y, groups)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    splits_dict[f'fold_{i+1}'] = {
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'train_participants': groups.iloc[train_index].unique().tolist(),
        'test_participants': groups.iloc[test_index].unique().tolist()
    }

    print(f"Fold {i+1}.")
    print(f"  - Treino com {len(X_train)} amostras de {len(groups.iloc[train_index].unique())} participantes.")
    print(f"  - Teste com {len(X_test)} amostras de {len(groups.iloc[test_index].unique())} participantes.")